# Сабмишн: поведение и история объявлений

Финальная сборка использует локальные поведенческие признаки и историю объявлений, доступную строго до начала окна каждой куки. CatBoost обучается на всём `train`, после чего строятся предсказания для `test`.

Целевая переменная тестовой части неизвестна, поэтому в этом ноутбуке нет оценки качества.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from artifacts import configure_logging
from config import CATBOOST_FINAL_PARAMS, RANDOM_SEED, ExperimentConfig
from data import load_metadata, load_or_prepare_events
from feature_sets import build_feature_set
from submission import make_submission

configure_logging()
pd.set_option("display.max_columns", 100)

c:\Users\Kolya\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Сборка локальных признаков

Здесь строится только часть, не использующая целевую переменную. История объявлений будет отдельно рассчитана внутри финальной процедуры: для обучающих строк — только по более ранним датам, для теста — по всему доступному прошлому `train`.

In [2]:
data = load_metadata()
events = load_or_prepare_events(data.metadata)
features, feature_spec = build_feature_set(
    "behavior_with_item_history",
    data.metadata,
    data.train,
    events,
)

pd.Series({
    "train_rows": len(data.train),
    "test_rows": len(data.test),
    "local_features": features.shape[1] - 1,
    "history_entities": ", ".join(feature_spec.history_entities),
    "history_alpha": feature_spec.history_alpha,
    "recency_half_life_days": feature_spec.recency_half_life_days,
    "random_seed": RANDOM_SEED,
}).to_frame("value")

16:40:32 | INFO | data | Метаданные загружены: train=11091, test=4909, ботов=899
16:40:32 | INFO | artifacts | События внутри окон: кеш, строк=288126, колонок=18
16:40:33 | INFO | feature_sets | Поведенческие наборы загружены из кеша
16:40:33 | INFO | artifacts | Блок признаков sequence: кеш, строк=16000, колонок=271
16:40:33 | INFO | artifacts | Блок признаков pointer_dynamics: кеш, строк=16000, колонок=82
16:40:33 | INFO | artifacts | Блок признаков missingness: кеш, строк=16000, колонок=97
16:40:33 | INFO | artifacts | Блок признаков text_hash: кеш, строк=16000, колонок=199


,value
train_rows,11091
test_rows,4909
local_features,705
history_entities,item
history_alpha,25.0
recency_half_life_days,5.0
random_seed,42


## Обучение и сохранение

Каждый запуск получает собственную папку `submissions/item_history/<дата_и_время>/`. Помимо `submission.csv` там остаются модель и все данные, необходимые для воспроизведения сборки.

In [3]:
final_config = ExperimentConfig(
    name="final_behavior_with_item_history",
    feature_set="behavior_with_item_history",
    model_name="catboost",
    model_params=CATBOOST_FINAL_PARAMS,
    use_frequency_features=True,
    frequency_feature_style="prefix",
    history_entities=feature_spec.history_entities,
    history_alpha=feature_spec.history_alpha,
    recency_half_life_days=feature_spec.recency_half_life_days,
    expected_feature_count=719,
    feature_version="compact_behavior_v1_with_local_blocks_v1",
    compute_mode="cpu",
)

run_dir, submission = make_submission(
    final_config,
    features,
    data.train,
    data.test,
    events,
    data.sample_submission,
    submission_group="item_history",
)

pd.Series({
    "submission_path": str(run_dir / "submission.csv"),
    "rows": len(submission),
    "missing_scores": int(submission["score"].isna().sum()),
    "min_score": submission["score"].min(),
    "max_score": submission["score"].max(),
}).to_frame("value")

0:	total: 185ms	remaining: 3m 41s
100:	total: 4.6s	remaining: 50s
200:	total: 8.54s	remaining: 42.4s
300:	total: 13.8s	remaining: 41.2s
400:	total: 19s	remaining: 37.9s
500:	total: 24.2s	remaining: 33.7s
600:	total: 30s	remaining: 29.9s
700:	total: 35.1s	remaining: 25s
800:	total: 40.2s	remaining: 20s
900:	total: 46.1s	remaining: 15.3s
1000:	total: 51.5s	remaining: 10.2s
1100:	total: 57s	remaining: 5.13s


16:41:38 | INFO | submission | Submission сохранён в C:\Users\Kolya\Desktop\ОтборНаСтажки\Авито\TestTask-ClassicML-3-avito-ds-bootcamp\submissions\item_history\20260928_164138\submission.csv


1199:	total: 1m 2s	remaining: 0us


,value
submission_path,C:\Users\Kolya\Desktop\ОтборНаСтажки\Авито\Tes...
rows,4909
missing_scores,0
min_score,0.000504
max_score,0.999751


## Результат

Модель обучена на всех 11 091 строках train с 719 признаками: 705 локальными и 14 колонками, добавленными при подготовке истории и частот. Использованы глубина 3, 1 200 деревьев и начальное число генератора 42; обучение прошло на процессоре.

Файл содержит 4 909 строк без пропусков. Оценки конечны и лежат в диапазоне от 0.000504 до 0.999751. Артефакты запуска сохранены в `submissions/item_history/20260928_164138/`.

Проверяющая система оценила финальный сабмишн в **0,86968** по Precision при Recall не ниже 70%. Добавление истории объявлений улучшило первый результат 0,81566 на 0,05402 абсолютного значения.